In [0]:
# This notebook demonstrates how to build a tool-calling agent using Unity Catalog hosted SQL functions in Databricks. It loads an electronics product dataset into a Delta table, creates a UC SQL function to look up product details by name and color, wraps that function as a LangChain tool via UCFunctionToolkit, and wires it into an AgentExecutor powered by a foundation model endpoint so users can ask natural language questions about products and get answers retrieved directly from the data.


In [0]:
%pip install databricks-langchain langchain-community langchain-experimental unitycatalog-ai[databricks] unitycatalog-langchain[databricks] "typing_extensions>=4.13.0"

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%sql
create volume if not exists genai_lab

In [0]:
df=spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/dbws01/default/genai_lab/electronics_products.csv")

In [0]:
from pyspark.sql.functions import *
from delta.tables import *

deltatablepath=f'/Volumes/dbws01/default/genai_lab/electronics_products'
df.write.format('delta').mode('overwrite').save(deltatablepath)

df.write.format('delta').saveAsTable('default.electronic_products')

In [0]:
#create unity catalog hosted functions client
from unitycatalog.ai.core.databricks import DatabricksFunctionClient
client=DatabricksFunctionClient()

/databricks/spark/python/databricks/connect/session.py:503: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


In [0]:
%sql
--sql hosted function

create or replace function default.lookup_electronics_item(
    product_name STRING COMMENT 'Name of the product to look up. for instant user query is how much does green webcam cost then the product name is green webcam',
    product_color STRING COMMENT 'Color of the product to look up'
)
RETURNS STRING
COMMENT "Return metadata about a specific product in the electronics items dataset, including its ID, price and description"
RETURN SELECT CONCAT(
'Product ID: ', productID, ', ',
'Product Name: ', productName, ', ',
'Product Colour: ', colour, ', ',
'Price: ', price
)
from default.electronic_products
where LOWER(productName) = LOWER(product_name)
and LOWER(colour) = LOWER(product_color)
LIMIT 1;


In [0]:
import typing_extensions
if hasattr(typing_extensions, '_TypedDictMeta') and not getattr(typing_extensions._TypedDictMeta, '_extra_items_patched', False):
    _orig_td_new = typing_extensions._TypedDictMeta.__new__
    def _patched_td_new(cls, *args, extra_items=None, **kwargs):
        return _orig_td_new(cls, *args, **kwargs)
    typing_extensions._TypedDictMeta.__new__ = staticmethod(_patched_td_new)
    typing_extensions._TypedDictMeta._extra_items_patched = True

from databricks_langchain import UCFunctionToolkit
func_name ="dbws01.default.lookup_electronics_item"
toolkit=UCFunctionToolkit(function_names=[func_name])
tools=toolkit.tools

/databricks/spark/python/databricks/connect/session.py:503: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


In [0]:
#adding tools to the agent
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from databricks_langchain import (ChatDatabricks, UCFunctionToolkit)
import mlflow

LLM_ENDPOINT_NAME='databricks-gemma-3-12b'
llm=ChatDatabricks(endpoint=LLM_ENDPOINT_NAME, temperature=0.1)
#define prompt with agent_scratchpad placeholder

prompt=ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant make sure to use tools for additional functionality."),
    ('human',"{input}"),
    ("placeholder", "{agent_scratchpad}")
])

mlflow.langchain.autolog()

agent=create_tool_calling_agent(llm, tools, prompt)

agent_executor=AgentExecutor(agent=agent, tools=tools, verbose=True)


In [0]:
#run agent
agent_executor.invoke({"input": "What is the price of the red graphics card"})



> Entering new AgentExecutor chain...

Invoking: `dbws01__default__lookup_electronics_item` with `{'product_name': 'graphics card', 'product_color': 'red'}`


{"format": "SCALAR", "value": "Product ID: P004, Product Name: Graphics Card, Product Colour: Red, Price: 152.8"}The price of the red graphics card is 152.8.

> Finished chain.


{'input': 'What is the price of the red graphics card',
 'output': 'The price of the red graphics card is 152.8.'}

Trace(trace_id='tr-8d93412454aca39fe9b6dd9e53abe3f9')



> Entering new AgentExecutor chain...
I do not have the capability to determine the number of graphics cards. I can look up metadata about specific electronics items if you provide a product name and color.


> Finished chain.


{'input': 'how many graphics cards are there?',
 'output': 'I do not have the capability to determine the number of graphics cards. I can look up metadata about specific electronics items if you provide a product name and color.\n'}

Trace(trace_id='tr-aad69ba82e06c370be3b5f7223d2ae8e')